In [ ]:
# 1. Import Required Libraries
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from PIL import Image

## 2. Load and Preprocess .pgm Data
Load .pgm images, preprocess, and prepare labels for training.

In [ ]:
# Set your data directory here
data_dir = '.'  # or the path where jam_bw_*Hz folders are located
img_height = 128  # adjust if needed
img_width = 256   # adjust if needed

# Find all jam_bw_*Hz folders
class_folders = [f for f in os.listdir(data_dir) if f.startswith('jam_bw_') and os.path.isdir(os.path.join(data_dir, f))]
class_folders.sort()  # ensure consistent label ordering
num_classes = len(class_folders)

# Load images and labels
X, y = [], []
for idx, folder in enumerate(class_folders):
    folder_path = os.path.join(data_dir, folder)
    for fname in os.listdir(folder_path):
        if fname.endswith('.pgm'):
            img_path = os.path.join(folder_path, fname)
            img = Image.open(img_path).resize((img_width, img_height))
            img = np.array(img)
            if img.ndim == 2:
                img = np.expand_dims(img, axis=-1)  # grayscale
            X.append(img)
            y.append(idx)
X = np.array(X)
y = np.array(y)
print(f"Loaded {X.shape[0]} images. Shape: {X.shape}")

## 3. Train/Test Split and Normalization
Split the data into train/test sets and normalize pixel values.

In [ ]:
# Normalize and split data
X = X.astype('float32') / 255.0
if X.shape[-1] == 1:
    X = np.repeat(X, 3, axis=-1)  # convert grayscale to 3 channels for CNN

y_cat = to_categorical(y, num_classes)
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 4. Build the CNN Model
Define and compile the convolutional neural network for classification.

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 5. Train the Model
Fit the model using the training and validation data.

In [ ]:
epochs = 20

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=32,
    validation_split=0.2
)

## 6. Evaluate and Visualize Results
Plot training history and print classification report.

In [ ]:
# Plot training history
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

# Evaluate on test set
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)
print('Classification Report')
print(classification_report(y_true, y_pred_classes, target_names=class_folders))

## 7. Predict on New .pgm Images
Function to classify a new .pgm spectrogram image.

In [ ]:
def predict_pgm(img_path):
    img = Image.open(img_path).resize((img_width, img_height))
    img = np.array(img)
    if img.ndim == 2:
        img = np.expand_dims(img, axis=-1)
    img = img.astype('float32') / 255.0
    img = np.repeat(img, 3, axis=-1)
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)
    class_idx = np.argmax(pred, axis=1)[0]
    class_label = class_folders[class_idx]
    return class_label

# Example usage:
# print(predict_pgm('jam_bw_200000Hz/example.pgm'))